# CreditWise — 03. Feature Engineering

Adds lending-domain features on top of the cleaned/encoded data: income aggregation, affordability ratios (loan-to-income, estimated EMI, EMI-to-income), cushion ratios (savings/collateral vs. loan size), and log transforms of skewed money columns.

**What changed from the original notebook** (see `src/feature_engineering.py` docstring for the full explanation):
- `Credit_Score` is no longer squared and overwritten in place — that destroyed the original column and didn't add real signal for a score that's already on a roughly linear risk scale.
- The column named `Applicant_Income_sq` was actually a log transform, not a square — renamed to `Log_Total_Income` so the name matches what it does.
- New features are grounded in lending domain logic (affordability, coverage) instead of arbitrary polynomial transforms of columns already in the model.
- Engineered features are **added alongside** the originals rather than replacing them, so you can compare each against the raw column and drop it later if it doesn't help.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.feature_engineering import engineer_features

In [ ]:
df = pd.read_csv("../data/processed/cleaned_encoded.csv")
df.shape

## Apply feature engineering

In [ ]:
df_fe = engineer_features(df)
new_cols = [c for c in df_fe.columns if c not in df.columns]
print("New columns added:", new_cols)
df_fe[new_cols].describe()

## Sanity check: how do the new features correlate with the target?

Worth checking before trusting a new feature — a ratio that's completely uncorrelated with the target is a sign something's off (e.g. divide-by-zero handling, wrong denominator).

In [ ]:
df_fe.select_dtypes(include="number").corr()["Loan_Approved"][new_cols].sort_values(ascending=False)

## Save for the modeling notebook

In [ ]:
OUT_PATH = "../data/processed/engineered.csv"
df_fe.to_csv(OUT_PATH, index=False)
print(f"Saved {df_fe.shape[0]} rows x {df_fe.shape[1]} cols to {OUT_PATH}")